# Customer Segmentation & Retention Analysis — E-Commerce
## Phase 3: Churn Prediction Model

**Model:** XGBoost binary classifier  
**Task:** Predict which customers will churn (Dormant / At-Risk = 1, Champions = 0)  
**Input features:** Recency, Frequency, Monetary (raw RFM values)  
**Experiment tracking:** MLflow (local SQLite backend)

---

## Section 1 — Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mlflow
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

CHURN_PATH    = '../data/processed/churn_results.csv'
CM_IMG        = '../data/processed/confusion_matrix.png'
ROC_IMG       = '../data/processed/roc_curve.png'
FI_IMG        = '../data/processed/feature_importance.png'
MLFLOW_URI    = 'sqlite:///../mlruns/mlflow.db'

print('Libraries loaded.')

In [ ]:
df = pd.read_csv(CHURN_PATH)

print(f'Shape     : {df.shape}')
print(f'Columns   : {df.columns.tolist()}')
print()

# Churn rate
churn_rate = df['churn_predicted'].mean() * 100
print(f'Predicted churn rate : {churn_rate:.1f}%')
print(df['Segment'].value_counts())
df.head()

---
## Section 2 — Model Performance

Five metrics are reported to give a complete picture of classifier performance.  
Accuracy alone is insufficient — in a dataset where 60% of customers churn, a model that predicts **everyone** churns would achieve 60% accuracy while being completely useless.

In [ ]:
# Display metrics from MLflow (or recreate from churn_results)
# These are the values logged to MLflow during src/churn_model.py execution

metrics = {
    'Metric'         : ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Score'          : [0.9830,     0.9873,      0.9846,   0.9860,     0.9993],
    'Business meaning': [
        '98.3% of all customers are correctly classified',
        '98.7% of flagged churners actually churn',
        '98.5% of actual churners are caught by the model',
        'Balanced average of precision and recall',
        'Near-perfect separation between retained and churned',
    ]
}
metrics_df = pd.DataFrame(metrics)
print('=== Model Performance Metrics ===')
print(metrics_df.to_string(index=False))

In [ ]:
# Display confusion matrix and ROC curve side by side

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, img_path, title in [
    (axes[0], CM_IMG,  'Confusion Matrix'),
    (axes[1], ROC_IMG, 'ROC Curve'),
]:
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.show()

**Business Interpretation — What these metrics mean for the retention team:**

| Metric | Score | What it means in practice |
|---|---|---|
| **Recall 98.5%** | 0.985 | Out of every 100 customers about to churn, we correctly identify **98–99 of them** in time to run a retention campaign. Almost no churner escapes detection. |
| **Precision 98.7%** | 0.987 | Out of every 100 customers we flag as churning, **98–99 actually will churn**. We waste almost no campaign budget on false alarms. |
| **ROC-AUC 0.999** | 0.9993 | The model **almost perfectly separates** retained customers from churned ones across all possible classification thresholds. |

**Why are the scores so high?**  
This level of performance is expected here — and it is important to understand why. The churn labels were derived directly from the RFM K-Means clusters in Phase 2. The Champions and Dormant/At-Risk groups have fundamentally different RFM profiles (50 vs 299 days recency; 12.8 vs 2.1 orders; £6,569 vs £613 spend). XGBoost is learning boundaries that are already clear in the feature space. In a production system, churn labels would be defined independently (e.g., 90-day inactivity), producing a more challenging — and realistic — classification problem. The architecture and evaluation framework established here applies directly to that harder task.

---
## Section 3 — Feature Importance

Understanding *why* the model makes predictions is as important as the predictions themselves — particularly when communicating results to non-technical stakeholders.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
img = mpimg.imread(FI_IMG)
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance values
fi_data = pd.DataFrame({
    'Feature'   : ['Frequency', 'Monetary', 'Recency'],
    'Importance': [0.5328,      0.2684,     0.1988],
})

fig = px.bar(
    fi_data,
    x='Importance', y='Feature',
    orientation='h',
    title='XGBoost Feature Importance — Churn Prediction',
    color='Importance',
    color_continuous_scale='Blues',
    template='plotly_white',
    height=320,
)
fig.update_layout(coloraxis_showscale=False, yaxis={'categoryorder': 'total ascending'})
fig.show()

**Business Insight — What drives churn prediction:**

**Frequency (53%) is the dominant predictor.** The number of distinct purchases a customer has made is the single strongest signal of whether they will stay or leave. This makes intuitive sense:

- A customer who has only ordered once is **overwhelmingly likely** to never return — they sampled the product but didn't form a habit.
- A customer with 5+ orders has demonstrated repeat intent — they have a purchasing pattern worth defending.

> **Actionable implication:** The highest-leverage intervention is converting first-time buyers into second-time buyers. A targeted follow-up email at day 14–21 post-purchase — perhaps with a personalised recommendation based on what they bought — could meaningfully shift the frequency distribution and reduce overall churn.

**Monetary (27%)** is the second strongest signal. Low-spend customers tend to churn at higher rates — they never found enough value to justify repeat visits. This suggests that customers whose first order is below a certain threshold (e.g., < £200) warrant a specific re-engagement nudge.

**Recency (20%)** is less dominant here because Frequency and Monetary already capture the same signal: a customer who orders rarely and spends little will also have poor recency by definition.

---
## Section 4 — Churn Probability Distribution

The churn probability score (0 → 1) is more useful than a binary label. It lets us **prioritise** which customers to contact first and size the retention budget appropriately.

In [ ]:
# Overall churn probability histogram

fig = px.histogram(
    df,
    x='churn_probability',
    nbins=50,
    title='Distribution of Churn Probabilities — All 5,878 Customers',
    labels={'churn_probability': 'Churn Probability', 'count': 'Number of Customers'},
    color_discrete_sequence=['steelblue'],
    template='plotly_white',
    height=380,
)
fig.update_layout(bargap=0.05)
# Add threshold lines
for threshold, color, label in [
    (0.8, 'red',    'Critical (0.8)'),
    (0.6, 'orange', 'High (0.6)'),
    (0.4, 'gold',   'Medium (0.4)'),
]:
    fig.add_vline(x=threshold, line_dash='dash', line_color=color,
                  annotation_text=label, annotation_position='top right')
fig.show()

In [ ]:
# Separate histograms per segment

fig = px.histogram(
    df,
    x='churn_probability',
    color='Segment',
    nbins=50,
    barmode='overlay',
    title='Churn Probability by Segment',
    labels={'churn_probability': 'Churn Probability'},
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Bold,
    template='plotly_white',
    height=380,
)
fig.update_layout(bargap=0.05)
fig.show()

In [ ]:
print('Churn probability statistics by segment:')
df.groupby('Segment')['churn_probability'].describe().round(3)

**Business Insight — Reading the probability distribution:**

The distribution is strongly **bimodal** — most customers score either very close to 0 or very close to 1, with relatively few in the 0.3–0.7 range. This reflects the clarity of separation between the two RFM profiles:

- **Champions** cluster tightly near probability 0 → the model is very confident they will not churn
- **Dormant / At-Risk** cluster tightly near probability 1 → the model is very confident they will churn

For practical campaign targeting, the small group of customers with **probabilities between 0.4 and 0.8** is the most interesting — these are borderline cases where a relatively small intervention (a timely email, a modest discount) could shift behaviour in either direction. The customers at probability > 0.9 have likely already decided not to return; heavy spend on them may have lower ROI than focusing on the 0.5–0.8 band.

---
## Section 5 — High Risk Customer Profiles

Identifying the highest-risk customers allows the retention team to prioritise personal outreach for those where the revenue-at-stake justifies the effort.

In [ ]:
# Top 20 highest churn probability customers

top20_risk = (
    df[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Segment', 'churn_probability']]
    .sort_values('churn_probability', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top20_risk.index += 1  # 1-based rank

print('Top 20 Customers by Churn Probability:')
top20_risk

In [ ]:
# Profile summary of top 100 highest-risk customers
top100 = df.nlargest(100, 'churn_probability')

print('Profile — Top 100 highest churn risk customers:')
print(f"  Avg Recency   : {top100['Recency'].mean():.0f} days")
print(f"  Avg Frequency : {top100['Frequency'].mean():.1f} orders")
print(f"  Avg Monetary  : £{top100['Monetary'].mean():,.0f}")
print(f"  Segment mix   :")
print(top100['Segment'].value_counts().to_string())
print(f"\n  Total revenue at stake: £{top100['Monetary'].sum():,.0f}")

**Business Insight — What the highest-risk customers look like:**

The top 100 highest-risk customers share a consistent profile:

- **Long recency:** They haven't purchased in several hundred days — well past the natural re-purchase window for this retailer
- **Very low frequency:** Most placed only 1–2 orders in the entire two-year period
- **Low monetary value:** Average spend is much lower than Champions

The key question for the business is: **is the revenue at stake worth the cost of intervention?** For customers who spent only £100–£200 total, an expensive personalised outreach campaign (phone call, dedicated account manager) is unlikely to be ROI-positive. For high-spend dormant customers — those with Monetary > £2,000 who simply went quiet — the case for investment is much stronger.

**Recommendation:** Filter the high-risk list by Monetary value and run a two-tier campaign:
- High-risk **+ high Monetary** → personal outreach, strong offer
- High-risk **+ low Monetary** → automated email sequence, light touch

---
## Section 6 — Business Decision Framework

A churn probability score only delivers value when it is translated into a **decision rule** that the marketing and CRM teams can act on. The table below defines four risk tiers with specific intervention strategies and expected ROI logic.

In [ ]:
# Assign risk tiers
def assign_tier(p):
    if p >= 0.8:   return 'Critical'
    elif p >= 0.6: return 'High'
    elif p >= 0.4: return 'Medium'
    else:          return 'Low'

df['risk_tier'] = df['churn_probability'].apply(assign_tier)

# Revenue at risk per tier
tier_summary = (
    df.groupby('risk_tier', as_index=False)
    .agg(
        Customers        =('CustomerID',        'count'),
        Avg_Monetary     =('Monetary',           'mean'),
        Total_Revenue    =('Monetary',           'sum'),
        Avg_Churn_Prob   =('churn_probability',  'mean'),
    )
)
tier_order = ['Critical', 'High', 'Medium', 'Low']
tier_summary['risk_tier'] = pd.Categorical(tier_summary['risk_tier'], categories=tier_order, ordered=True)
tier_summary = tier_summary.sort_values('risk_tier').reset_index(drop=True)
tier_summary['Pct_Customers'] = (tier_summary['Customers'] / tier_summary['Customers'].sum() * 100).round(1)
tier_summary['Avg_Monetary'] = tier_summary['Avg_Monetary'].round(0).astype(int)
tier_summary['Total_Revenue'] = tier_summary['Total_Revenue'].round(0).astype(int)
tier_summary['Avg_Churn_Prob'] = tier_summary['Avg_Churn_Prob'].round(3)

print('Risk Tier Summary:')
tier_summary

In [ ]:
# Visualise customers and revenue at risk by tier

tier_colors = {'Critical': '#d62728', 'High': '#ff7f0e', 'Medium': '#ffbf00', 'Low': '#2ca02c'}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Customers per Risk Tier', 'Total Revenue at Risk per Tier (£)']
)

for col, y_col, fmt in [
    (1, 'Customers',     ',.0f'),
    (2, 'Total_Revenue', '£,.0f'),
]:
    fig.add_trace(
        go.Bar(
            x=tier_summary['risk_tier'],
            y=tier_summary[y_col],
            marker_color=[tier_colors[t] for t in tier_summary['risk_tier']],
            showlegend=False,
            text=[f'{v:,}' if col == 1 else f'£{v:,}' for v in tier_summary[y_col]],
            textposition='outside',
        ),
        row=1, col=col,
    )

fig.update_layout(
    title_text='Churn Risk Tiers — Customer Distribution & Revenue at Stake',
    height=420,
    template='plotly_white',
)
fig.show()

**Business Decision Framework — Where to focus the retention budget:**

| Risk Tier | Churn Prob | Customers | Revenue at Risk | Recommended Action | Expected ROI |
|---|---|---|---|---|---|
| **Critical** | 0.8 – 1.0 | ~3,442 | Highest share | Win-back email + time-limited discount (e.g. 20% off, 7-day expiry). For high-Monetary subset (>£2K): personal account outreach. | High if targeted by Monetary; low if blanketed |
| **High** | 0.6 – 0.8 | ~100 | Moderate | Targeted re-engagement email with personalised product recommendations. Moderate incentive (10–15% discount). | Medium — borderline customers, responsive to nudges |
| **Medium** | 0.4 – 0.6 | ~50 | Low–Moderate | Automated nurture sequence (2–3 emails, no discount, value-led content). Monitor for movement toward Critical. | Medium — still salvageable with light touch |
| **Low** | 0.0 – 0.4 | ~2,286 | Revenue concentrated in Champions | Standard loyalty programme, no special spend. These are your Champions — invest in deepening the relationship, not rescuing it. | High — they are already retained |

---

**Where should the business focus first?**

The retention budget should be allocated in three priority buckets:

1. **Critical tier, high Monetary** — These are dormant customers who historically spent significant amounts (>£2,000). Even a 10% win-back conversion on this subset generates material incremental revenue. This is the highest-ROI intervention.

2. **High tier (0.6–0.8)** — A small group (~100 customers) who are genuinely on the fence. They are the most responsive to a well-timed, personalised offer. The probability range suggests they haven't fully disengaged.

3. **Critical tier, low Monetary** — Large volume but lower individual value. An automated, low-cost email sequence is appropriate. Do not over-invest with discounts — the economics rarely justify it for customers with a total lifetime spend under £200.

---
## Section 7 — MLflow Experiment Summary

Every model run is tracked in a local MLflow database so that results are reproducible and comparable across iterations.

In [ ]:
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri(MLFLOW_URI)

# Load all runs from the churn_prediction experiment
runs = mlflow.search_runs(
    experiment_names=['churn_prediction'],
    order_by=['start_time DESC'],
)

display_cols = [
    'run_id',
    'params.n_estimators', 'params.max_depth', 'params.learning_rate',
    'metrics.accuracy', 'metrics.precision', 'metrics.recall',
    'metrics.f1', 'metrics.roc_auc',
    'start_time',
]
available = [c for c in display_cols if c in runs.columns]
print('MLflow Experiment: churn_prediction')
print(f'Total runs logged: {len(runs)}')
runs[available].head(5)

In [ ]:
# Print the most recent run details cleanly
if len(runs) > 0:
    latest = runs.iloc[0]
    run_id = latest['run_id']
    print(f'Most recent run ID : {run_id}')
    print()
    print('Hyperparameters:')
    param_cols = [c for c in runs.columns if c.startswith('params.')]
    for col in param_cols:
        print(f'  {col.replace("params.", ""):<20}: {latest[col]}')
    print()
    print('Metrics:')
    metric_cols = [c for c in runs.columns if c.startswith('metrics.')]
    for col in metric_cols:
        print(f'  {col.replace("metrics.", ""):<20}: {latest[col]:.4f}')

**Why experiment tracking matters in production:**

MLflow solves a problem that every data science team eventually faces: **"what exactly produced that model?"**

Without tracking:
- A model trained in a notebook last month cannot be reproduced — the hyperparameters are forgotten, the data version is unknown
- Comparing two model iterations requires manually reading through code and output logs
- Deploying the "best" model is a guess

With MLflow tracking, every run captures:
- **Parameters** — exactly which hyperparameters were used
- **Metrics** — all evaluation scores at the time of training
- **Model artifact** — the serialised model weights, ready for deployment
- **Timestamp** — when the run occurred, enabling rollback if a new model regresses

In a team environment, this means a senior data scientist can review a junior's model run, reproduce it exactly, and compare it against a baseline — all without needing to re-run code or share Jupyter notebooks.

---

## Next Steps — Phase 4: Customer Lifetime Value

The churn probability scores produced here feed directly into Phase 4:
- Customers with **low churn probability (Champions)** are the primary candidates for CLV estimation — we want to know how much they are worth so we can justify retention investment
- The **BG/NBD + Gamma-Gamma** model will estimate predicted purchases and average order value per customer over the next 12–24 months
- Combining CLV × (1 - churn probability) gives a **risk-adjusted expected value** per customer — the ultimate input to retention budget allocation